# CAZ Framework — What Happens Inside a Transformer?

*A layer-by-layer dissection of how a language model constructs semantic concepts*

When a transformer processes text, concepts like *negation*, *causation*, and *certainty* are not passively stored — they are built up across layers, in specific regions, through a measurable geometric process. The **Concept Allocation Zone (CAZ)** framework makes that process visible.

This notebook uses Qwen2.5-7B as a case study and works through the core ideas from first principles: the single-layer probing problem, the three CAZ signals, zone anatomy, multimodal allocation, and how the underlying concept *direction* evolves alongside the amplitude. No GPU required — all results are pre-computed.

**[github.com/jamesrahenry/Rosetta](https://github.com/jamesrahenry/Rosetta)** · Henry (2026a–b)

In [1]:
import subprocess, sys

def _pip(*pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])

try:
    _pip("rosetta_tools>=1.3.1")
except subprocess.CalledProcessError:
    _pip("rosetta_tools @ git+https://github.com/jamesrahenry/Rosetta_Tools.git@v1.3.1")

_pip("huggingface_hub", "matplotlib", "numpy", "scipy", "scikit-learn")


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
from huggingface_hub import hf_hub_download

HF_REPO = "james-ra-henry/Rosetta-Activations"

# All results in this notebook use a single model
PRIMARY_MODEL = "Qwen/Qwen2.5-7B"   # 28 layers, 3584-dim, GQA
PRIMARY_LABEL = "Qwen2.5-7B"

# Concepts available in the Rosetta Activations dataset
ALL_CONCEPTS = [
    "temporal_order", "causation", "agency",
    "negation", "sentiment", "moral_valence",
    "urgency", "certainty", "credibility",
    "sarcasm", "specificity",
]

def model_key(model_id: str) -> str:
    return model_id.replace("/", "_").replace("-", "_")

print("Setup complete.")

In [3]:
from rosetta_tools.viz_style import (
    concept_color, CONCEPT_COLORS, FAMILY_COLORS, THEME, apply_theme,
)
import matplotlib.pyplot as plt

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 10,
    "axes.titlesize": 11,
    "axes.labelsize": 10,
    "legend.fontsize": 9,
    "figure.dpi": 150,
    "figure.facecolor": "white",
})
print("Style loaded.")

Style loaded.


## Download pre-computed CAZ results

Pre-computed results for Qwen2.5-7B across eleven concepts are stored in the [Rosetta Activations](https://huggingface.co/datasets/james-ra-henry/Rosetta-Activations) dataset on Hugging Face. Each file contains, for every layer: Fisher separation $S(l)$, coherence $C(l)$, velocity $v(l)$, and the dominant concept direction $\hat{d}(l)$.

In [ ]:
caz = {}
for concept in ALL_CONCEPTS:
    filename = f"models/{model_key(PRIMARY_MODEL)}/caz_{concept}.json"
    path = hf_hub_download(HF_REPO, filename=filename, repo_type="dataset")
    with open(path) as f:
        caz[concept] = json.load(f)

d0 = caz[ALL_CONCEPTS[0]]
print(f"Model:    {PRIMARY_LABEL}  ({d0['n_layers']} layers, {d0['hidden_dim']}-dim)")
print(f"Concepts: {len(caz)}  ({', '.join(ALL_CONCEPTS)})")

## 1. The single-layer problem

The standard probing approach picks a layer — often the middle layer or the layer with the best validation accuracy — and trains a classifier there. The implicit assumption is that there is some consistently good layer to pick.

The chart below plots $S(l)$, a measure of how linearly separable positive and negative examples are, for all eleven concepts across every layer of Qwen2.5-7B. Every concept has a distinct peak at a different depth. Probe at layer 10 and you capture some concepts well; probe at layer 20 and you capture different ones. There is no universally good layer — the right depth is concept-specific.

In [ ]:
n_layers = caz[ALL_CONCEPTS[0]]["n_layers"]

fig, ax = plt.subplots(figsize=(12, 5))
fig.patch.set_facecolor("white")

for concept in ALL_CONCEPTS:
    metrics = caz[concept]["layer_data"]["metrics"]
    depth   = [m["layer"] / n_layers * 100 for m in metrics]
    sep     = [m["separation_fisher"] for m in metrics]
    peak    = caz[concept]["layer_data"]["peak_depth_pct"]
    color   = concept_color(concept)
    ax.plot(depth, sep, color=color, lw=1.8, alpha=0.85, label=concept.replace("_", " "))
    ax.axvline(peak, color=color, lw=0.8, ls=":", alpha=0.35)

ax.set_xlabel("Depth (% of layers)", fontsize=11)
ax.set_ylabel("S(l) — Separation", fontsize=11)
ax.set_title(f"Every concept peaks at a different depth — {PRIMARY_LABEL}\n"
             "Dotted lines mark each peak; no single layer is universally good",
             fontsize=12, fontweight="bold")
ax.set_xlim(0, 100)
ax.legend(fontsize=7.5, ncol=2, loc="upper left", framealpha=0.9)
apply_theme(ax)
plt.tight_layout()
plt.show()

## 2. The three signals

The CAZ framework tracks concept construction through three layer-wise signals:

| Signal | Symbol | What it measures |
|--------|--------|-----------------|
| **Separation** | $S(l)$ | Fisher-normalised centroid distance — how far apart positive and negative class representations are, scaled by within-class scatter |
| **Coherence** | $C(l)$ | Mean cosine similarity to class centroid — how geometrically organised each class is internally |
| **Velocity** | $v(l)$ | Rate of change of $S(l)$ — peaks just *before* the separation peak, marking the onset of assembly |

Separation tells you *where* the concept is strongest. Coherence tells you whether that separation is a clean organised signal or diffuse scatter. Velocity is the leading edge — it marks where the model starts actively constructing the representation, not just where it has finished.

In [ ]:
concept = "causation"
d = caz[concept]
metrics = d["layer_data"]["metrics"]
depth_x = [m["layer"] / d["n_layers"] * 100 for m in metrics]

SIG_COLORS = {"sep": "#1565C0", "coh": "#2E7D32", "vel": "#E65100"}

fig, axes = plt.subplots(3, 1, figsize=(10, 7), sharex=True)
fig.patch.set_facecolor("white")
fig.suptitle(f"{PRIMARY_LABEL} — '{concept}'", fontsize=13, fontweight="bold")

for ax, key, ylabel, color in zip(
    axes,
    ["separation_fisher", "coherence", "velocity"],
    ["S(l) — Separation", "C(l) — Coherence", "v(l) — Velocity"],
    [SIG_COLORS["sep"], SIG_COLORS["coh"], SIG_COLORS["vel"]],
):
    vals = [m[key] for m in metrics]
    ax.plot(depth_x, vals, color=color, lw=2)
    ax.axhline(0, color=THEME["spine"], lw=0.8, ls="--")
    ax.set_ylabel(ylabel, fontsize=10)
    apply_theme(ax)

peak_pct = d["layer_data"]["peak_depth_pct"]
for ax in axes:
    ax.axvline(peak_pct, color="#C62828", lw=1.5, ls=":", alpha=0.7)
axes[0].annotate(f"peak {peak_pct:.0f}%",
                 xy=(peak_pct, axes[0].get_ylim()[1]),
                 xytext=(peak_pct + 3, axes[0].get_ylim()[1] * 0.85),
                 color="#C62828", fontsize=9)
axes[-1].set_xlabel("Depth (% of layers)", fontsize=11)
plt.tight_layout()
plt.show()

## 3. Zone anatomy — more than a peak

The peak of $S(l)$ is the centre of a Concept Allocation Zone, but the zone has *extent*: a **start** (where velocity first rises above threshold, signalling the onset of assembly) and an **end** (where separation drops back off and the construction phase is over). The width of a zone carries information — a narrow zone means the concept is assembled quickly and precisely; a wide zone means assembly is spread across many layers.

`find_caz_regions` from `rosetta_tools` detects the full extent of each zone rather than just the peak.

In [ ]:
from rosetta_tools.caz import LayerMetrics, find_caz_regions

concept = "causation"
d = caz[concept]
metrics = d["layer_data"]["metrics"]
n_layers = d["n_layers"]
depth_x = [m["layer"] / n_layers * 100 for m in metrics]
sep     = [m["separation_fisher"] for m in metrics]

lm = [LayerMetrics(layer=m["layer"], separation=m["separation_fisher"],
                   coherence=m["coherence"], velocity=m["velocity"]) for m in metrics]
profile = find_caz_regions(lm)
region  = profile.dominant

s_pct = region.start / n_layers * 100
e_pct = region.end   / n_layers * 100
color = concept_color(concept)

fig, ax = plt.subplots(figsize=(11, 4))
fig.patch.set_facecolor("white")

ax.plot(depth_x, sep, color=color, lw=2.2, zorder=3)
ax.fill_between(depth_x, sep, alpha=0.08, color=color, zorder=2)
ax.axvspan(s_pct, e_pct, alpha=0.18, color=color, zorder=1)
ax.axvline(s_pct,              color=color, lw=1.2, ls="--", alpha=0.7)
ax.axvline(region.depth_pct,   color=color, lw=2.0, ls="-",  alpha=0.9)
ax.axvline(e_pct,              color=color, lw=1.2, ls="--", alpha=0.7)

ymax = max(sep) * 1.18
ax.text(s_pct,            ymax * 0.96, "start", color=color, fontsize=9, ha="center")
ax.text(region.depth_pct, ymax * 0.96, "peak",  color=color, fontsize=9, ha="center", fontweight="bold")
ax.text(e_pct,            ymax * 0.96, "end",   color=color, fontsize=9, ha="center")

ax.set_xlabel("Depth (% of layers)", fontsize=11)
ax.set_ylabel("S(l) — Separation", fontsize=11)
ax.set_ylim(0, ymax)
ax.set_xlim(0, 100)
ax.set_title(f"CAZ zone anatomy — '{concept}' in {PRIMARY_LABEL}", fontsize=12, fontweight="bold")
apply_theme(ax)
plt.tight_layout()
plt.show()
print(f"Zone: {s_pct:.0f}% → {region.depth_pct:.0f}% (peak) → {e_pct:.0f}%   width = {e_pct-s_pct:.0f} pp")

## 4. Multimodal allocation

Not every concept assembles in a single step. Some show **two distinct peaks** — typically a shallow one (early syntactic or structural processing) and a deeper one (later semantic integration). A single-layer probe positioned at either peak entirely misses the other assembly event.

`find_caz_regions` returns each peak separately. `agency` in Qwen2.5-7B is a clear example: two zones with peaks at 25% and 75% of model depth — one early, one late, separated by a 50-layer gap with relatively low separation in between.

In [ ]:
ZONE_COLORS = ["#1565C0", "#E65100"]

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
fig.patch.set_facecolor("white")
fig.suptitle(f"Unimodal vs. multimodal allocation — {PRIMARY_LABEL}",
             fontsize=12, fontweight="bold")

for ax, concept in zip(axes, ["causation", "agency"]):
    d = caz[concept]
    metrics = d["layer_data"]["metrics"]
    n_layers = d["n_layers"]
    depth_x = [m["layer"] / n_layers * 100 for m in metrics]
    sep     = [m["separation_fisher"] for m in metrics]

    lm = [LayerMetrics(layer=m["layer"], separation=m["separation_fisher"],
                       coherence=m["coherence"], velocity=m["velocity"]) for m in metrics]
    profile = find_caz_regions(lm)

    color = concept_color(concept)
    ax.plot(depth_x, sep, color=color, lw=2.2, zorder=3)
    ax.fill_between(depth_x, sep, alpha=0.08, color=color)

    for i, region in enumerate(profile.regions):
        s_pct = region.start / n_layers * 100
        e_pct = region.end   / n_layers * 100
        zc = ZONE_COLORS[i % len(ZONE_COLORS)]
        ax.axvspan(s_pct, e_pct, alpha=0.18, color=zc, zorder=1,
                   label=f"CAZ {i+1}  (peak {region.depth_pct:.0f}%)")
        ax.axvline(region.depth_pct, color=zc, lw=1.5, ls="--", alpha=0.8)

    mm = "  ★ multimodal" if profile.is_multimodal else ""
    ax.set_title(f"'{concept}' — {profile.n_regions} zone{'s' if profile.n_regions > 1 else ''}{mm}",
                 fontsize=11, fontweight="bold")
    ax.set_xlabel("Depth (% of layers)", fontsize=10)
    ax.set_ylabel("S(l) — Separation", fontsize=10)
    ax.set_xlim(0, 100)
    ax.legend(fontsize=9, loc="upper left")
    apply_theme(ax)

plt.tight_layout()
plt.show()

## 5. GEM — the direction also evolves

The three signals above track *amplitude*: how strongly a concept is geometrically expressed at each layer. The **Geometric Evolution Map (GEM)** tracks the other half of the story: which *direction* in activation space the concept occupies at each layer.

The dominant concept direction $\hat{d}(l)$ is the unit vector pointing from the negative-class centroid to the positive-class centroid. It is not static — it rotates as the representation is refined layer by layer. Measuring the layer-to-layer cosine $\cos(\hat{d}_l,\, \hat{d}_{l-1})$ gives a **direction stability** signal: values near 1.0 mean the concept direction barely changed; a sharp drop marks a **handoff point** where the geometric basis shifts.

The two panels below reveal a structural asymmetry:
- **Separation** (left) peaks at different depths for each concept — amplitude is concept-specific
- **Direction stability** (right) dips at the *same* layers for all concepts — the handoff structure is a property of the model, not the concept

In [ ]:
DEMO_CONCEPTS = ["causation", "agency", "negation", "certainty"]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor("white")
fig.suptitle(f"Amplitude vs. direction evolution — {PRIMARY_LABEL}",
             fontsize=12, fontweight="bold")

nl = caz[ALL_CONCEPTS[0]]["n_layers"]
depth_x = [l / nl * 100 for l in range(nl)]

for concept in DEMO_CONCEPTS:
    color   = concept_color(concept)
    metrics = caz[concept]["layer_data"]["metrics"]
    sep     = [m["separation_fisher"] for m in metrics]
    vecs    = [np.array(m["dom_vector"]) for m in metrics]

    dir_cos = [np.nan]   # no predecessor at layer 0
    for i in range(1, len(vecs)):
        v0, v1 = vecs[i-1], vecs[i]
        cos = float(np.dot(v0, v1) / (np.linalg.norm(v0) * np.linalg.norm(v1) + 1e-12))
        dir_cos.append(cos)

    label = concept.replace("_", " ")
    axes[0].plot(depth_x, sep,     color=color, lw=1.8, alpha=0.85, label=label)
    axes[1].plot(depth_x, dir_cos, color=color, lw=1.8, alpha=0.85, label=label)

axes[0].set_title("Separation — concept-specific peaks",       fontsize=11)
axes[0].set_ylabel("S(l) — Separation",                        fontsize=11)
axes[1].set_title("Direction stability — shared model structure", fontsize=11)
axes[1].set_ylabel("cos(d̂ₗ, d̂ₗ₋₁)",                          fontsize=11)

for ax in axes:
    ax.set_xlabel("Depth (% of layers)", fontsize=11)
    ax.set_xlim(0, 100)
    ax.legend(fontsize=9, loc="upper left")
    apply_theme(ax)

plt.tight_layout()
plt.show()

## 6. The full picture — all concepts in one model

With the vocabulary established — zones, multimodal allocation, direction evolution — here is the complete concept topology of Qwen2.5-7B across all eleven concepts. Each row is one concept; the bar spans the zone's start-to-end extent; the dot marks the peak. Concepts with two bars assembled in two distinct events.

This is a portrait of a *single model's* internal structure, not a cross-architecture claim.

In [ ]:
concept_order = sorted(ALL_CONCEPTS, key=lambda c: caz[c]["layer_data"]["peak_depth_pct"])

bar_h = 0.28
fig, ax = plt.subplots(figsize=(13, len(concept_order) * 0.65 + 1))
fig.patch.set_facecolor("white")

for ci, concept in enumerate(concept_order):
    d = caz[concept]
    n_layers = d["n_layers"]
    metrics = d["layer_data"]["metrics"]
    lm = [LayerMetrics(layer=m["layer"], separation=m["separation_fisher"],
                       coherence=m["coherence"], velocity=m["velocity"]) for m in metrics]
    profile = find_caz_regions(lm)
    color = concept_color(concept)

    for i, region in enumerate(profile.regions):
        s_pct = region.start / n_layers * 100
        e_pct = region.end   / n_layers * 100
        alpha = 0.80 if i == 0 else 0.45   # second zone slightly faded
        ax.barh(ci, e_pct - s_pct, left=s_pct, height=bar_h,
                color=color, alpha=alpha, zorder=2)
        ax.plot(region.depth_pct, ci, "o", color=color, ms=6, zorder=3,
                markeredgecolor="white", markeredgewidth=0.6)

ax.set_yticks(range(len(concept_order)))
ax.set_yticklabels([c.replace("_", " ") for c in concept_order], fontsize=10)
ax.set_xlabel("Depth (% of layers)", fontsize=11)
ax.set_xlim(0, 100)
ax.set_title(f"Concept topology — {PRIMARY_LABEL}\n"
             "Bar = zone extent  ·  dot = peak  ·  faded bar = second zone (multimodal)",
             fontsize=12, fontweight="bold")
apply_theme(ax)
plt.tight_layout()
plt.show()

## Summary

| Section | What we covered |
|---------|----------------|
| 1. Single-layer problem | Every concept peaks at a different depth; no fixed layer is universally useful |
| 2. Three signals | $S(l)$, $C(l)$, $v(l)$ — amplitude, organisation, and rate of change |
| 3. Zone anatomy | CAZ zones have start, peak, and end; width reflects how precisely a concept is localised |
| 4. Multimodal allocation | Some concepts assemble in two events; a single probe misses one of them |
| 5. GEM | The concept direction rotates layer-by-layer; handoff points are model-structural, not concept-specific |
| 6. Concept topology | A full portrait of one model's internal concept landscape |

---

### Next notebooks

| Notebook | What it covers |
|----------|---------------|
| `02_caz_interactive_demo.ipynb` | Load Qwen2.5-7B (4-bit) and run CAZ on your own concept pairs |
| `03_caz_implementation_demo.ipynb` | Implement the metrics from scratch; reproduce the cross-architecture convergence result from Paper 4 |

### Papers and resources

| | |
|-|-|
| Paper 1 — CAZ Framework | [Henry 2026a](https://arxiv.org/abs/PLACEHOLDER) |
| Paper 2 — GEM | [Henry 2026b](https://arxiv.org/abs/PLACEHOLDER) |
| `rosetta_tools` | [GitHub](https://github.com/jamesrahenry/Rosetta_Tools) |
| Concept pairs dataset | [Rosetta_Concept_Pairs](https://github.com/jamesrahenry/Rosetta_Concept_Pairs) |